In [21]:

# ==========================================================
# CELL 2: IMPORT REQUIRED LIBRARIES
# ==========================================================
#
# Libraries used for:
# - Audio capture
# - Speech detection
# - Speech recognition
# - Gemini interaction
# - Text-to-speech
#
# ==========================================================
import os
import re
import time
import queue
import subprocess

import torch
import sounddevice as sd
import numpy as np

from collections import deque
from scipy.io.wavfile import write

from faster_whisper import WhisperModel

from silero_vad import (
    load_silero_vad,
    VADIterator
)

from dotenv import load_dotenv
from groq import Groq

In [22]:
# ==========================================================
# CELL 3: SYSTEM CONFIGURATION AND MODEL LOADING
# ==========================================================
#
# Initializes:
# - Groq LLM (Llama 3.3 70B)
# - Faster-Whisper Small
# - Silero VAD
#
# Audio Settings:
# - Sample Rate = 16000 Hz
# - Block Size = 512
#
# ==========================================================

# Load environment variables from .env file
load_dotenv()

# ==========================================================
# GROQ CONFIGURATION
# ==========================================================

from groq import Groq

GROQ_API_KEY = os.getenv(
    "GROQ_API_KEY"
)

client = Groq(
    api_key=GROQ_API_KEY
)

MODEL_NAME = "llama-3.3-70b-versatile"

print(f"Groq Model Loaded: {MODEL_NAME}")

# ==========================================================
# AUDIO CONFIGURATION
# ==========================================================

SAMPLE_RATE = 16000
BLOCK_SIZE = 512

PRE_SPEECH_SECONDS = 0.5

# ==========================================================
# LOAD FASTER WHISPER
# ==========================================================

print("Loading Faster Whisper...")

asr_model = WhisperModel(
    "medium",
    device="cuda",
    compute_type="float16"
)

print("Faster Whisper Loaded")

# ==========================================================
# LOAD SILERO VAD
# ==========================================================

print("Loading Silero VAD...")

vad_model = load_silero_vad()

vad_iterator = VADIterator(
    vad_model,
    sampling_rate=SAMPLE_RATE,
    min_silence_duration_ms=800
)

print("Silero VAD Loaded")

Groq Model Loaded: llama-3.3-70b-versatile
Loading Faster Whisper...
Faster Whisper Loaded
Loading Silero VAD...
Silero VAD Loaded


In [23]:
# ==========================================================
# CELL 4: CONTINUOUS MICROPHONE CAPTURE
# ==========================================================
#
# Creates:
# - Audio Queue
# - Audio Callback
# - Input Stream
#
# Audio frames are continuously pushed into a queue
# for VAD processing.
#
# ==========================================================
audio_queue = queue.Queue()

def audio_callback(
    indata,
    frames,
    time_info,
    status
):

    if status:
        if getattr(status, "input_overflow", False):
            print("Warning: audio input overflow")
        else:
            print(status)

    audio_queue.put(
        indata.copy()
    )

pre_buffer = deque(
    maxlen=int(
        (SAMPLE_RATE * PRE_SPEECH_SECONDS)
        / BLOCK_SIZE
    )
)

print("Microphone Ready")


Microphone Ready


In [24]:
# ==========================================================
# CELL 5: SPEECH SEGMENTATION AND TRANSCRIPTION
# ==========================================================
#
# Functions:
#
# 1. collect_speech_chunk()
#    - Detect speech start
#    - Collect speech audio
#    - Detect speech end
#
# 2. transcribe_chunk()
#    - Convert speech to text
#    - Measure ASR latency
#
# ==========================================================
def collect_speech_chunk():

    global pre_buffer

    collected_audio = []

    speech_active = False

    # Clear any stale queued audio before starting a new capture.
    try:
        while True:
            audio_queue.get_nowait()
    except queue.Empty:
        pass

    print("\nListening...")

    with sd.InputStream(
        samplerate=SAMPLE_RATE,
        channels=1,
        dtype="float32",
        blocksize=BLOCK_SIZE,
        callback=audio_callback
    ):

        while True:

            chunk = audio_queue.get()

            audio = chunk.flatten()

            pre_buffer.append(audio)

            result = vad_iterator(
                torch.tensor(
                    audio,
                    dtype=torch.float32
                ),
                return_seconds=True
            )

            if result is not None:

                if (
                    "start" in result
                    and not speech_active
                ):

                    speech_active = True

                    collected_audio = list(
                        pre_buffer
                    )

                    print("Speech Started")

                elif (
                    "end" in result
                    and speech_active
                ):

                    print("Speech Ended")

                    speech_active = False

                    audio_chunk = np.concatenate(
                        collected_audio
                    )

                    vad_iterator.reset_states()

                    pre_buffer.clear()

                    return audio_chunk

            if speech_active:

                collected_audio.append(
                    audio
                )


def transcribe_chunk(audio_chunk):

    write(
        "temp_chunk.wav",
        SAMPLE_RATE,
        (
            audio_chunk * 32767
        ).astype(np.int16)
    )

    start_time = time.time()

    segments, _ = asr_model.transcribe(
        "temp_chunk.wav",
        language="en",
        beam_size=5
    )

    transcript = " ".join(
        segment.text
        for segment in segments
    ).strip()

    latency = (
        time.time() - start_time
    )

    return transcript, latency

In [25]:
# ==========================================================
# CELL 6: RESPONSE GENERATION USING GROQ
# ==========================================================

def ask_groq(transcript):

    prompt = f"""
    You are a voice assistant.

    Rules:
    - Keep responses concise
    - Maximum 3 sentences
    - Natural conversational style

    User:
    {transcript}
    """

    try:

        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=0.7,
            max_tokens=150
        )

        return (
            response
            .choices[0]
            .message
            .content
        )

    except Exception as e:

        print(
            f"Groq Error: {e}"
        )

        return (
            "Sorry, I am unable to generate "
            "a response right now."
        )

In [26]:
# ==========================================================
# CELL 7: RESPONSE CLEANING
# ==========================================================
#
# Removes:
# - Markdown symbols
# - Formatting characters
# - Extra whitespace
#
# Produces text suitable for speech synthesis.
#
# ==========================================================
def clean_text(text):

    text = re.sub(
        r'[*_`#>-]+',
        ' ',
        text
    )

    text = re.sub(
        r'\[(.*?)\]\((.*?)\)',
        r'\1',
        text
    )

    text = re.sub(
        r'\s+',
        ' ',
        text
    ).strip()

    return text

In [27]:
# ==========================================================
# CELL 8: RESPONSE LOGGING
# ==========================================================
#
# Stores the latest Gemini response in a text file.
#
# This file is later used as input for
# Piper Text-to-Speech synthesis.
#
# ==========================================================
def save_response(text):

    with open(
        "response.txt",
        "w",
        encoding="utf-8"
    ) as f:

        f.write(text)

In [28]:
# ==========================================================
# Cell 9: STREAMING PIPER TTS
# ==========================================================

def stream_tts(text):

    try:

        print("\nGenerating Streaming Speech...")

        tts_start = time.time()

        process = subprocess.Popen(
            [
                "piper/piper.exe",
                "--model",
                "piper/en_US-amy-medium.onnx",
                "--config",
                "piper/en_US-amy-medium.onnx.json",
                "--output-raw"
            ],
            stdin=subprocess.PIPE,
            stdout=subprocess.PIPE,
            stderr=subprocess.DEVNULL
        )

        process.stdin.write(
            text.encode("utf-8")
        )

        process.stdin.close()

        SAMPLE_RATE_TTS = 22050
        CHUNK_BYTES = 4096

        first_audio_played = False
        ttfb = None

        stream = sd.RawOutputStream(
            samplerate=SAMPLE_RATE_TTS,
            channels=1,
            dtype="int16"
        )

        stream.start()

        while True:

            chunk = process.stdout.read(
                CHUNK_BYTES
            )

            if not chunk:
                break

            if not first_audio_played:

                ttfb = (
                    time.time()
                    - tts_start
                )

                first_audio_played = True

            stream.write(chunk)

        stream.stop()
        stream.close()

        process.stdout.close()
        process.wait()

        total_synthesis_time = (
            time.time()
            - tts_start
        )

        return (
            ttfb,
            total_synthesis_time
        )

    except Exception as e:

        print(
            f"TTS Error: {e}"
        )

        return (
            None,
            None
        )

In [29]:
# ==========================================================
# Cell 10: MAIN VOICE ASSISTANT LOOP
# ==========================================================

print("\nVoice Assistant Started")

while True:

    print("\n=== New Interaction ===")
    print("Listening for next speech...")

    audio_chunk = collect_speech_chunk()

    duration = (
        len(audio_chunk)
        / SAMPLE_RATE
    )

    if duration < 0.5:

        print(
            "Detected too-short audio chunk."
        )

        continue

    chunk_end_time = time.time()

    transcript, asr_latency = (
        transcribe_chunk(
            audio_chunk
        )
    )

    if not transcript.strip():

        continue

    total_latency = (
        time.time()
        - chunk_end_time
    )

    print("\n" + "=" * 60)

    print("\nUSER:")
    print(transcript)

    print(
        f"\nASR Latency: "
        f"{total_latency:.2f} sec"
    )

    normalized_transcript = re.sub(
        r'[^a-z0-9 ]+',
        ' ',
        transcript.lower()
    ).strip()

    stop_phrases = [
        "stop",
        "exit",
        "quit",
        "goodbye",
        "stop assistant",
        "please stop",
        "stop please"
    ]

    if any(
        normalized_transcript == phrase
        or normalized_transcript.startswith(
            phrase + " "
        )
        or normalized_transcript.endswith(
            " " + phrase
        )
        or (
            " " + phrase + " "
        ) in (
            " "
            + normalized_transcript
            + " "
        )
        for phrase in stop_phrases
    ):

        print(
            "\nAssistant stopped..."
        )

        break

    print(
        "\nGenerating Groq Response..."
    )

    llm_start_time = time.time()

    assistant_response = (
        ask_groq(
            transcript
        )
    )

    llm_latency = (
        time.time()
        - llm_start_time
    )

    assistant_response = (
        clean_text(
            assistant_response
        )
    )

    print("\nASSISTANT:")
    print("-" * 60)

    print(
        assistant_response
    )

    print("-" * 60)

    save_response(
        assistant_response
    )

    # ==================================================
    # STREAM TTS
    # ==================================================

    ttfb, total_synthesis_time = (
        stream_tts(
            assistant_response
        )
    )

    # ==================================================
    # LOG EVERYTHING TO FILE
    # ==================================================

    with open(
        "metrics_log.txt",
        "a",
        encoding="utf-8"
    ) as f:

        f.write("\n")
        f.write("=" * 70 + "\n")

        f.write(
            f"Timestamp: "
            f"{time.strftime('%Y-%m-%d %H:%M:%S')}\n"
        )

        f.write(
            f"USER: "
            f"{transcript}\n\n"
        )

        f.write(
            f"ASSISTANT: "
            f"{assistant_response}\n\n"
        )

        f.write(
            f"ASR Latency: "
            f"{total_latency:.2f} sec\n"
        )

        f.write(
            f"LLM Latency: "
            f"{llm_latency:.2f} sec\n"
        )

        if ttfb is not None:

            f.write(
                f"TTS TTFB: "
                f"{ttfb:.3f} sec\n"
            )

            f.write(
                f"TTS Total Synthesis Time: "
                f"{total_synthesis_time:.3f} sec\n"
            )

        else:

            f.write(
                "TTS Metrics Unavailable\n"
            )

        f.write(
            "=" * 70 + "\n"
        )

    # ==================================================
    # DISPLAY METRICS
    # ==================================================

    print("\n" + "=" * 60)

    print("PERFORMANCE METRICS")

    print("=" * 60)

    print(
        f"ASR Latency              : "
        f"{total_latency:.2f} sec"
    )

    print(
        f"LLM Latency              : "
        f"{llm_latency:.2f} sec"
    )

    if ttfb is not None:

        print(
            f"TTS TTFB                : "
            f"{ttfb:.3f} sec"
        )

        print(
            f"TTS Total Synthesis Time: "
            f"{total_synthesis_time:.3f} sec"
        )

    else:

        print(
            "TTS Metrics Unavailable"
        )

    print("=" * 60)

    print(
        "\nMetrics saved to metrics_log.txt"
    )

    time.sleep(3)


Voice Assistant Started

=== New Interaction ===
Listening for next speech...

Listening...
Speech Started
Speech Ended


USER:
What is RAG pipeline?

ASR Latency: 6.20 sec

Generating Groq Response...

ASSISTANT:
------------------------------------------------------------
The RAG pipeline, or Retrieval Augmented Generation pipeline, is a type of natural language processing framework. It combines information retrieval and generation to produce more accurate and informative responses. This approach helps AI models like me provide better answers by retrieving relevant information from a database before generating a response.
------------------------------------------------------------

Generating Streaming Speech...

PERFORMANCE METRICS
ASR Latency              : 6.20 sec
LLM Latency              : 0.55 sec
TTS TTFB                : 1.464 sec
TTS Total Synthesis Time: 25.509 sec

Metrics saved to metrics_log.txt

=== New Interaction ===
Listening for next speech...

Listening...
Speech